# AAA-ASR — Kaggle Public Dataset Fine-Tuning (Experiment 2: Frozen Encoder v1.2)

**Author:** Nandan (Member 5 — AAA-ASR)  
**Date:** 14 November 2026  
**Repository:** `Speech-Synthesis/AAA_Baselines`  
**Architecture:** Native OpenAI Whisper Multilingual `base` ($V=51,865$, 74M params)  
**Training Strategy:** Frozen Audio Encoder (Decoder Fine-Tuning Only ~37.2M params)  
**Checkpoints Contract:** Baseline (`zero_shot_baseline.pt`), Best (`public_best.pt`), Final (`public_final.pt`)  

---

### Experiment 2 Governance & Invariants:
1. **Acoustic Feature Preservation:** Audio encoder is strictly frozen (`encoder.eval()`, `requires_grad=False`).
2. **Prefix-Space Alignment:** Transcripts tokenized as `" " + text.lower()` matching Whisper BPE formatting.
3. **FP16 Scaler Synchronization:** `scheduler.step()` and update counter advance ONLY upon successful optimizer step.
4. **Independent Checkpointing:** `best_finetuned_wer` initialized to $\infty$, capturing true fine-tuned checkpoints.
5. **Calibration Evaluation:** `public_cal.jsonl` evaluated with average logprob confidence metrics.
6. **Held-Out Test Governance:** Evaluated as a subsequent test evaluation (after Experiment 1 baseline exposure).


## 1. Environment & Pre-Flight Hardware Verification

In [ ]:
!pip install --quiet openai-whisper soundfile scipy

import torch
import sys
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Total VRAM: {vram:.2f} GB")
    assert vram >= 6.0, "Minimum 6GB VRAM required for Whisper Base fine-tuning."
else:
    print("WARNING: Running on CPU. Training will be slow!")


## 2. Dataset Staging: OpenSLR Acquisition & Linux-Compatible Manifest Generation

Downloads LibriSpeech `dev-clean` (337.9 MB) from official OpenSLR SLR12, extracts audio, and generates
the 4 speaker-disjoint manifest splits (`train`, `val`, `cal`, `test`) with 100% Linux-compatible paths.
Reproduces the exact 24/6/4/6 speaker partitioning from `manifest_generator.py`.

In [ ]:
import os
import sys
import tarfile
import urllib.request
import json
import re
import unicodedata
from pathlib import Path
from collections import defaultdict
import soundfile as sf

FILLER_WORDS = frozenset({"um", "uh", "er", "hmm", "erm", "ah"})
DIGIT_TO_WORD = {
    "0": "zero", "1": "one", "2": "two", "3": "three", "4": "four",
    "5": "five", "6": "six", "7": "seven", "8": "eight", "9": "nine",
    "10": "ten", "11": "eleven", "12": "twelve", "13": "thirteen",
    "14": "fourteen", "15": "fifteen", "16": "sixteen", "17": "seventeen",
    "18": "eighteen", "19": "nineteen", "20": "twenty", "21": "twenty one",
    "30": "thirty", "40": "forty", "50": "fifty", "60": "sixty",
    "70": "seventy", "80": "eighty", "90": "ninety",
}
WORD_ALIASES = {"oh": "zero"}
SPELLING_MAP = {
    "colour": "color", "favourite": "favorite", "grey": "gray",
    "licence": "license", "defence": "defense", "catalogue": "catalog",
    "centre": "center", "metre": "meter", "litre": "liter",
    "analyse": "analyze", "organise": "organize", "recognise": "recognize",
    "realise": "realize", "apologise": "apologize",
}
CONTRACTIONS = {
    "can't": "cannot", "won't": "will not", "don't": "do not",
    "doesn't": "does not", "didn't": "did not", "isn't": "is not",
    "aren't": "are not", "wasn't": "was not", "weren't": "were not",
    "haven't": "have not", "hasn't": "has not", "hadn't": "had not",
    "couldn't": "could not", "wouldn't": "would not", "shouldn't": "should not",
    "i'm": "i am", "you're": "you are", "we're": "we are",
    "they're": "they are", "he's": "he is", "she's": "she is",
    "it's": "it is", "i've": "i have", "you've": "you have",
    "we've": "we have", "they've": "they have", "i'll": "i will",
    "you'll": "you will", "we'll": "we will", "they'll": "they will",
    "i'd": "i would", "you'd": "you would", "we'd": "we would",
    "they'd": "they would", "let's": "let us",
}

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text).lower().strip()
    for contraction, expansion in CONTRACTIONS.items():
        text = re.sub(r"\b" + re.escape(contraction) + r"\b", expansion, text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()
    words = [w for w in text.split() if w not in FILLER_WORDS]
    normalized = []
    for word in words:
        if word in DIGIT_TO_WORD:
            normalized.extend(DIGIT_TO_WORD[word].split())
        else:
            normalized.append(word)
    words = [SPELLING_MAP.get(w, w) for w in normalized]
    words = [WORD_ALIASES.get(w, w) for w in words]
    return " ".join(words)

DATA_ROOT = Path("./data").resolve()
DATA_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE_PATH = DATA_ROOT / "dev-clean.tar.gz"
EXTRACT_DIR = DATA_ROOT / "LibriSpeech" / "dev-clean"
OPENSLR_URL = "https://www.openslr.org/resources/12/dev-clean.tar.gz"

if not EXTRACT_DIR.exists():
    if not ARCHIVE_PATH.exists():
        print(f"Downloading LibriSpeech dev-clean from {OPENSLR_URL} ...")
        try:
            import subprocess
            cmd = ["curl", "-L", "-C", "-", "-o", str(ARCHIVE_PATH), OPENSLR_URL]
            subprocess.run(cmd, check=True)
        except Exception as e:
            print(f"curl failed ({e}); falling back to urllib.request ...")
            urllib.request.urlretrieve(OPENSLR_URL, str(ARCHIVE_PATH))
        print(f"Download complete. Size: {ARCHIVE_PATH.stat().st_size / (1024*1024):.2f} MB")
    print(f"Extracting archive to {DATA_ROOT} ...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall(path=DATA_ROOT)
    print("Archive extraction complete.")
else:
    print(f"Extracted LibriSpeech dataset already present at: {EXTRACT_DIR}")

candidates = list(DATA_ROOT.glob("**/dev-clean"))
assert len(candidates) > 0, f"Failed to locate dev-clean directory under {DATA_ROOT}"
dev_clean_dir = candidates[0]
print(f"Using dev-clean audio source directory: {dev_clean_dir}")

transcripts = {}
for trans_file in dev_clean_dir.glob("*/*/*.trans.txt"):
    with open(trans_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split(" ", 1)
                if len(parts) == 2:
                    transcripts[parts[0]] = parts[1].strip()

flac_files = sorted(list(dev_clean_dir.glob("*/*/*.flac")))
print(f"Discovered {len(flac_files)} FLAC audio files and {len(transcripts)} transcripts.")
assert len(flac_files) == 2703, f"Expected 2,703 FLAC files, found {len(flac_files)}"

speaker_utts = defaultdict(list)
for af in flac_files:
    utt_id = af.stem
    speaker_id = utt_id.split("-")[0]
    text = transcripts.get(utt_id, "")
    assert text, f"Missing transcript for {utt_id}"
    speaker_utts[speaker_id].append({
        "utt_id": utt_id,
        "path": af,
        "transcript": text,
        "speaker_id": speaker_id
    })

sorted_speakers = sorted(speaker_utts.keys())
assert len(sorted_speakers) == 40, f"Expected 40 unique speakers, found {len(sorted_speakers)}"

train_speakers = set(sorted_speakers[:24])
val_speakers = set(sorted_speakers[24:30])
cal_speakers = set(sorted_speakers[30:34])
test_speakers = set(sorted_speakers[34:40])

splits = {"train": train_speakers, "val": val_speakers, "cal": cal_speakers, "test": test_speakers}
all_spk_sets = [train_speakers, val_speakers, cal_speakers, test_speakers]
snames = ["train", "val", "cal", "test"]
for i in range(len(all_spk_sets)):
    for j in range(i + 1, len(all_spk_sets)):
        overlap = all_spk_sets[i].intersection(all_spk_sets[j])
        assert not overlap, f"Speaker leakage between {snames[i]} and {snames[j]}: {overlap}"

split_manifests = {"train": [], "val": [], "cal": [], "test": []}
all_sample_ids = set()

for split_name, spk_set in splits.items():
    for spk in sorted(spk_set):
        for item in speaker_utts[spk]:
            p = item["path"].resolve()
            info = sf.info(str(p))
            duration = float(info.duration)
            sample_rate = int(info.samplerate)
            assert sample_rate == 16000, f"Unexpected sample rate: {sample_rate}"
            raw_text = item["transcript"]
            norm_text = normalize_text(raw_text)
            sample_id = f"librispeech_dev_clean_{item['utt_id']}"
            assert sample_id not in all_sample_ids, f"Duplicate sample_id: {sample_id}"
            all_sample_ids.add(sample_id)
            posix_audio_path = p.as_posix()
            assert not posix_audio_path.startswith("E:") and not posix_audio_path.startswith("C:"), \
                f"Windows drive letter detected in audio_path: {posix_audio_path}"
            rec = {
                "sample_id": sample_id,
                "dataset_name": "LibriSpeech",
                "dataset_version": "dev-clean (OpenSLR SLR12)",
                "audio_path": posix_audio_path,
                "transcript": raw_text,
                "normalized_transcript": norm_text,
                "speaker_id": f"speaker_{spk}",
                "duration_seconds": round(duration, 3),
                "sample_rate": sample_rate,
                "split": split_name,
                "licence": "CC BY 4.0",
                "is_synthetic": False
            }
            split_manifests[split_name].append(rec)

for split_name in ["train", "val", "cal", "test"]:
    mpath = Path(f"public_{split_name}.jsonl")
    with open(mpath, "w", encoding="utf-8") as f:
        for rec in split_manifests[split_name]:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Wrote {mpath.name}: {len(split_manifests[split_name])} records.")

assert len(split_manifests["train"]) == 1544
assert len(split_manifests["val"]) == 433
assert len(split_manifests["cal"]) == 299
assert len(split_manifests["test"]) == 427
assert len(all_sample_ids) == 2703
print("\n>>> ALL PRE-TRAINING DATA STAGING GATES PASSED <<<\n")


## 3. Metrics Implementation (WER, CER, EPA)

In [ ]:
def compute_levenshtein(seq1, seq2):
    n1, n2 = len(seq1), len(seq2)
    dp = [[0] * (n2 + 1) for _ in range(n1 + 1)]
    for i in range(n1 + 1): dp[i][0] = i
    for j in range(n2 + 1): dp[0][j] = j
    for i in range(1, n1 + 1):
        for j in range(1, n2 + 1):
            cost = 0 if seq1[i - 1] == seq2[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[n1][n2]

def compute_metrics(predictions, references):
    total_w_dist, total_ref_words = 0, 0
    total_c_dist, total_ref_chars = 0, 0
    exact_matches = 0
    for p, r in zip(predictions, references):
        norm_p = normalize_text(p)
        norm_r = normalize_text(r)
        if norm_p == norm_r: exact_matches += 1
        pw, rw = norm_p.split(), norm_r.split()
        total_w_dist += compute_levenshtein(pw, rw)
        total_ref_words += max(len(rw), 1)
        total_c_dist += compute_levenshtein(norm_p, norm_r)
        total_ref_chars += max(len(norm_r), 1)
    n = len(predictions)
    return {
        "wer": round((total_w_dist / total_ref_words) * 100.0, 2),
        "cer": round((total_c_dist / total_ref_chars) * 100.0, 2),
        "exact_phrase_accuracy": round((exact_matches / n) * 100.0, 2) if n > 0 else 0.0,
        "total": n
    }


## 4. Dataset Loading with Resampling & Leading-Space Lowercase Tokenization

In [ ]:
import numpy as np
import scipy.signal
import whisper
from whisper.tokenizer import get_tokenizer
from torch.utils.data import Dataset, DataLoader

class WhisperSpeechDataset(Dataset):
    def __init__(self, manifest_path, tokenizer):
        self.records = []
        with open(manifest_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    self.records.append(json.loads(line))
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        audio, sr = sf.read(rec["audio_path"], dtype="float32")
        if audio.size == 0:
            raise ValueError(f"Empty audio file: {rec['audio_path']}")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != 16000:
            target_len = int(round(len(audio) * 16000.0 / sr))
            audio = scipy.signal.resample(audio, target_len).astype(np.float32)
            sr = 16000

        audio = whisper.pad_or_trim(audio)
        mel = whisper.log_mel_spectrogram(audio)

        # Prepend leading space and lowercase text matching Whisper BPE prefix formatting
        clean_text = " " + rec["transcript"].strip().lower()
        prompt_tokens = list(self.tokenizer.sot_sequence_including_notimestamps)
        text_tokens = self.tokenizer.encode(clean_text)
        tokens = prompt_tokens + text_tokens + [self.tokenizer.eot]

        dec_input = torch.tensor(tokens[:-1], dtype=torch.long)
        target = torch.tensor(tokens[1:], dtype=torch.long)
        target[: len(prompt_tokens) - 1] = -100

        return {"mel": mel, "dec_input": dec_input, "target": target, "transcript": rec["transcript"], "sample_id": rec["sample_id"]}

def make_collate_fn(tokenizer):
    pad_token_id = tokenizer.eot
    def collate_fn(batch):
        mels = torch.stack([item["mel"] for item in batch])
        max_len = max(len(item["dec_input"]) for item in batch)
        dec_inputs, targets = [], []
        for item in batch:
            pad = max_len - len(item["dec_input"])
            dec_inputs.append(torch.cat([item["dec_input"], torch.full((pad,), pad_token_id, dtype=torch.long)]))
            targets.append(torch.cat([item["target"], torch.full((pad,), -100, dtype=torch.long)]))
        return {
            "mel": mels,
            "dec_input": torch.stack(dec_inputs),
            "target": torch.stack(targets),
            "transcripts": [item["transcript"] for item in batch],
            "sample_ids": [item["sample_id"] for item in batch]
        }
    return collate_fn


## 5. Model Loading & Audio Encoder Freezing (Experiment 2 Strategy)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading OpenAI Whisper multilingual base model on {device}...")
model = whisper.load_model("base", device=device)

# 1. Strictly freeze the audio encoder to prevent catastrophic forgetting
model.encoder.eval()
for param in model.encoder.parameters():
    param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable parameters (Decoder only): {trainable_params:,} ({trainable_params/74000000*100:.1f}%)")
print(f"Frozen parameters (Audio Encoder):   {frozen_params:,} ({frozen_params/74000000*100:.1f}%)")

tokenizer = get_tokenizer(multilingual=True, language="en", task="transcribe")
collate_fn = make_collate_fn(tokenizer)

def export_native_whisper_checkpoint(model, output_path):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    checkpoint = {
        "dims": model.dims.__dict__,
        "model_state_dict": model.state_dict()
    }
    torch.save(checkpoint, output_path)
    print(f"Exported native Whisper checkpoint to: {output_path} ({Path(output_path).stat().st_size / (1024**2):.2f} MB)")


## 6. Evaluation Loop with Confidence Metrics

In [ ]:
def evaluate_split(model, dataloader, device, record_confidence=False):
    model.eval()
    preds, refs = [], []
    avg_logprobs = []
    with torch.no_grad():
        for batch in dataloader:
            mels = batch["mel"].to(device)
            options = whisper.DecodingOptions(
                language="en",
                task="transcribe",
                without_timestamps=True,
                beam_size=1,
                fp16=(device.type == "cuda")
            )
            res = model.decode(mels, options)
            for r, ref in zip(res, batch["transcripts"]):
                preds.append(r.text)
                refs.append(ref)
                if record_confidence:
                    avg_logprobs.append(r.avg_logprob)
    metrics = compute_metrics(preds, refs)
    if record_confidence and avg_logprobs:
        mean_lp = float(np.mean(avg_logprobs))
        metrics["mean_avg_logprob"] = round(mean_lp, 4)
        metrics["mean_confidence"] = round(float(np.exp(mean_lp)), 4)
        metrics["high_confidence_ratio"] = round(float(np.mean([1.0 if lp > -0.5 else 0.0 for lp in avg_logprobs])) * 100.0, 2)
    return metrics


## 7. Experiment 2 Training Execution (LR=5e-6, Scaler Synchronized, 3 Checkpoints)

In [ ]:
import math
import time

DATA_DIR = Path("./")
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_ds = WhisperSpeechDataset(DATA_DIR / "public_train.jsonl", tokenizer)
val_ds = WhisperSpeechDataset(DATA_DIR / "public_val.jsonl", tokenizer)
cal_ds = WhisperSpeechDataset(DATA_DIR / "public_cal.jsonl", tokenizer)
test_ds = WhisperSpeechDataset(DATA_DIR / "public_test.jsonl", tokenizer)

# Assert strict test set isolation
test_sample_ids = set(r["sample_id"] for r in test_ds.records)
assert len(set(r["sample_id"] for r in train_ds.records).intersection(test_sample_ids)) == 0
assert len(set(r["sample_id"] for r in val_ds.records).intersection(test_sample_ids)) == 0
assert len(set(r["sample_id"] for r in cal_ds.records).intersection(test_sample_ids)) == 0
print("Strict test split isolation verified: zero test data in train/val/cal sets.")

batch_size = 8
grad_accum = 2
max_epochs = 3

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
cal_loader = DataLoader(cal_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

batches_per_epoch = len(train_loader)
updates_per_epoch = math.ceil(batches_per_epoch / grad_accum)
total_optimizer_steps = updates_per_epoch * max_epochs
warmup_steps = int(round(total_optimizer_steps * 0.12))

# Checkpoint 1/3: Export Zero-Shot Pretrained Baseline
baseline_ckpt_path = OUTPUT_DIR / "whisper_base_zero_shot_baseline.pt"
export_native_whisper_checkpoint(model, baseline_ckpt_path)

# Zero-shot baseline evaluation
val_baseline = evaluate_split(model, val_loader, device)
print(f"\n[Stage 0 Baseline Val] WER: {val_baseline['wer']}% | EPA: {val_baseline['exact_phrase_accuracy']}%")

# Optimizer on trainable decoder parameters only with 5e-6 learning rate
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-6, weight_decay=0.01)

def lr_lambda(step):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_optimizer_steps - warmup_steps))
    return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.0)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

# Best fine-tuned metric initialized independently to infinity
best_finetuned_wer = float("inf")
best_ckpt_path = OUTPUT_DIR / "whisper_base_public_best.pt"
final_ckpt_path = OUTPUT_DIR / "whisper_base_public_final.pt"

# Training loop
model.train()
model.encoder.eval()  # Keep encoder strictly in eval mode
global_step = 0
successful_updates = 0
skipped_updates = 0
accum_count = 0
accum_loss = 0.0
optimizer.zero_grad()

for epoch in range(max_epochs):
    print(f"\n>>> Epoch {epoch + 1}/{max_epochs} <<<")
    for batch_idx, batch in enumerate(train_loader):
        mels = batch["mel"].to(device)
        dec_inputs = batch["dec_input"].to(device)
        targets = batch["target"].to(device)

        is_last = (batch_idx + 1 == batches_per_epoch)
        is_accum_step = ((batch_idx + 1) % grad_accum == 0) or is_last

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(mels, dec_inputs)
            raw_loss = loss_fn(logits.view(-1, logits.size(-1)), targets.view(-1))

        accum_count += 1
        accum_loss += raw_loss.item()
        loss = raw_loss / (accum_count if is_accum_step else grad_accum)
        scaler.scale(loss).backward()

        if is_accum_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), 1.0)

            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            scale_after = scaler.get_scale()

            # Step scheduler and advance counter ONLY when optimizer update succeeded
            if scale_before <= scale_after:
                successful_updates += 1
                scheduler.step()
                global_step += 1
            else:
                skipped_updates += 1
                print(f"  [AMP WARNING] Step skipped due to inf/nan gradients! Scale: {scale_before:.1f} -> {scale_after:.1f}")

            avg_loss = accum_loss / accum_count
            accum_loss = 0.0
            accum_count = 0
            optimizer.zero_grad()

            if global_step % 20 == 0 or global_step == 1:
                lr_val = optimizer.param_groups[0]["lr"]
                print(f"  Step {global_step:3d}/{total_optimizer_steps} | Loss: {avg_loss:.4f} | LR: {lr_val:.2e} | Updates: {successful_updates} | Skipped: {skipped_updates}")

            if global_step > 0 and global_step % 50 == 0:
                val_metrics = evaluate_split(model, val_loader, device)
                print(f"  [Val Step {global_step}] WER: {val_metrics['wer']}% | EPA: {val_metrics['exact_phrase_accuracy']}%")
                if val_metrics['wer'] < best_finetuned_wer:
                    best_finetuned_wer = val_metrics['wer']
                    print(f"  [Checkpoint 2/3] -> New best fine-tuned model (WER: {best_finetuned_wer}%)! Saving checkpoint...")
                    export_native_whisper_checkpoint(model, best_ckpt_path)
                model.train()
                model.encoder.eval()

# Checkpoint 3/3: Export Final Epoch Checkpoint
print("\n--- Exporting Final Epoch Checkpoint (Checkpoint 3/3) ---")
export_native_whisper_checkpoint(model, final_ckpt_path)


## 8. Calibration Confidence Analysis & Baseline vs. Fine-Tuned Test Comparison

In [ ]:
eval_model_path = best_ckpt_path if best_ckpt_path.exists() else final_ckpt_path
print(f"Loading model for final evaluation: {eval_model_path.name}")
eval_model = whisper.load_model(str(eval_model_path), device=device)

# 1. Calibration Set Confidence Analysis
print("\n--- Evaluating Calibration Split (Confidence Analysis) ---")
cal_metrics = evaluate_split(eval_model, cal_loader, device, record_confidence=True)
print(f"Calibration Set: WER={cal_metrics['wer']}%, Mean Logprob={cal_metrics.get('mean_avg_logprob')}, Mean Confidence={cal_metrics.get('mean_confidence')}, High-Conf Ratio={cal_metrics.get('high_confidence_ratio')}%")

# 2. Held-Out Test Evaluation (Subsequent Test Evaluation)
print("\n--- Evaluating Held-Out Test Split (Subsequent Evaluation) ---")
baseline_model = whisper.load_model(str(baseline_ckpt_path), device=device)
base_test_metrics = evaluate_split(baseline_model, test_loader, device)
fine_test_metrics = evaluate_split(eval_model, test_loader, device)

wer_delta = round(fine_test_metrics['wer'] - base_test_metrics['wer'], 2)
print(f"Baseline Test WER:   {base_test_metrics['wer']}%")
print(f"Fine-Tuned Test WER: {fine_test_metrics['wer']}%")
print(f"WER Delta:           {wer_delta:+0.2f}%")

summary = {
    "experiment": "Experiment 2 — Frozen Audio Encoder v1.2",
    "governance": {
        "test_evaluation_count": 2,
        "notes": "Experiment 1 recorded baseline zero-shot test evaluation. Experiment 2 evaluates the fine-tuned model as subsequent evaluation. Test split was strictly held out from training and checkpoint selection."
    },
    "training_dynamics": {
        "total_steps_planned": total_optimizer_steps,
        "successful_optimizer_updates": successful_updates,
        "skipped_optimizer_updates": skipped_updates
    },
    "checkpoints": {
        "zero_shot_baseline": str(baseline_ckpt_path.name),
        "best_fine_tuned": str(best_ckpt_path.name) if best_ckpt_path.exists() else None,
        "final_epoch": str(final_ckpt_path.name)
    },
    "validation": {
        "zero_shot_baseline_wer": val_baseline['wer'],
        "best_fine_tuned_wer": best_finetuned_wer
    },
    "calibration": cal_metrics,
    "held_out_test_comparison": {
        "baseline_test_metrics": base_test_metrics,
        "finetuned_test_metrics": fine_test_metrics,
        "wer_delta": wer_delta
    }
}

with open(OUTPUT_DIR / "PUBLIC_EXPERIMENT_METRICS.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("\n[Complete] PUBLIC_EXPERIMENT_METRICS.json saved successfully!")
